# Menu Assistant — Qwen3-4B QLoRA SFT scaffold

Status: **prepared, not executed on a GPU**. No model has been downloaded, trained, called, or exported by this scaffold. The 36 synthetic examples demonstrate the contract; they are not a production training corpus or evidence of model quality.

Train learns bounded decisions and context grounding. Menu prices, stock and product identifiers come from each request's runtime context. Dietary/ingredient/portion metadata is `UNREVIEWED`; those requests refer to staff.

Upload the complete `ai/menu-assistant` folder (including `data` and `training`) or open this notebook in the repository. Colab/Kaggle must have Internet and a compatible NVIDIA GPU only when you explicitly enable installation/model steps. Do not add customer conversations, keys or cookies.


In [ ]:
from pathlib import Path
import importlib.util
import json
import re

# Every expensive/network/model action is opt-in. Run data validation first.
RUN_INSTALL = False
RUN_TRAINING = False
GENERATE_PREDICTIONS = False
RUN_EXPORT = False

MODEL_ID = "unsloth/Qwen3-4B-unsloth-bnb-4bit"
BASE_MODEL_ID = "Qwen/Qwen3-4B"
MODEL_REVISION = ""  # REQUIRED: exact 40-character model repository commit before model actions.
MAX_SEQ_LENGTH = 4096  # Includes the shared policy plus up to eight structured fixture products.
SEED = 3407
INFERENCE_VARIANT = "ADAPTER"  # BASE: pinned base without LoRA; ADAPTER: trained/loaded LoRA.
ADAPTER_PATH = None  # Existing local adapter directory for prediction/export without this session's training.
RUN_LABEL = "manual-run"  # Use a new label for each experiment; existing evidence is not overwritten.

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd() / "ai/menu-assistant"]
BASE = next((path.resolve() for path in candidates if (path / "data/train.jsonl").is_file() and (path / "training/validate_data.py").is_file()), None)
if BASE is None:
    raise FileNotFoundError("Keep data/ and training/ together; run from the repository or menu-assistant folder.")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", RUN_LABEL):
    raise ValueError("RUN_LABEL must be a simple folder name.")
RUN_DIR = BASE / "training/artifacts" / RUN_LABEL
module_spec = importlib.util.spec_from_file_location("menu_assistant_preflight", BASE / "training/validate_data.py")
preflight = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(preflight)
DATA_REPORT = preflight.validate_dataset(BASE)
print(json.dumps(DATA_REPORT, ensure_ascii=False, indent=2))
if not DATA_REPORT["contractPolicyChecked"]:
    print("Shared TypeScript policy file is absent in this hosted copy; run the repository's TS validator before training.")


## Candidate dependencies

The installation cell is separate and disabled. The profile follows the official Qwen3 conversational notebook's Transformers/TRL/Datasets pins with declared dependencies; Unsloth releases are candidates, **not a tested compatibility guarantee**. The selected runtime owns CUDA, PyTorch and Triton. Inspect resolver output and `pip check`, then restart the kernel if installation changed imported packages and rerun validation. Save the actual resolved environment in each run manifest. No automatic hosted experiment tracker or upload is configured.


In [ ]:
import subprocess
import sys

if RUN_INSTALL:
    assert DATA_REPORT["total"] > 0
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(BASE / "training/requirements-colab.txt")])
    subprocess.check_call([sys.executable, "-m", "pip", "check"])
    print("Restart the kernel if needed, then rerun configuration and validation before model cells.")
else:
    print("Installation disabled; data preflight used only the Python standard library.")


## Reproducible run manifest and model guard

An exact model repository revision is mandatory before model actions. Model loading can download several GB. The original Qwen3-4B and its matching 4-bit conversion are the target; do not substitute Base, Thinking-2507 or Instruct-2507 without revisiting the template. Adapter reuse requires an actual adapter directory, preventing accidental export or scoring of an untouched base model.


In [ ]:
import datetime
import hashlib
import importlib.metadata
import platform

ACTIVE_MODEL_STEP = RUN_TRAINING or GENERATE_PREDICTIONS or RUN_EXPORT
model = tokenizer = trainer = None
manifest = None
trained_in_this_session = False
loaded_existing_adapter = False
loaded_base_model = False

if ACTIVE_MODEL_STEP:
    assert DATA_REPORT["total"] > 0
    if INFERENCE_VARIANT not in ("BASE", "ADAPTER"):
        raise ValueError("INFERENCE_VARIANT must be BASE or ADAPTER.")
    if INFERENCE_VARIANT == "BASE" and (RUN_TRAINING or RUN_EXPORT or ADAPTER_PATH is not None):
        raise ValueError("BASE prediction must not train/load/export an adapter; use a separate ADAPTER run.")
    if not re.fullmatch(r"[0-9a-fA-F]{40}", MODEL_REVISION):
        raise ValueError("Set MODEL_REVISION to the exact Hugging Face model repository commit; do not use a floating branch.")
    # Import Unsloth before transformers/TRL so its runtime patches are applied.
    from unsloth import FastLanguageModel, is_bfloat16_supported
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Select a compatible NVIDIA CUDA GPU runtime before model actions.")
    if (RUN_DIR / "run-manifest.json").exists():
        raise FileExistsError("Choose a new RUN_LABEL to preserve the existing experiment manifest.")
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    packages = {}
    for package in ("unsloth", "unsloth_zoo", "torch", "triton", "transformers", "trl", "datasets", "peft", "accelerate", "bitsandbytes", "huggingface_hub"):
        try: packages[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError: packages[package] = None
    manifest = {
        "status": "manual-run-started",
        "startedAt": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "modelId": MODEL_ID, "modelRevision": MODEL_REVISION, "baseModelId": BASE_MODEL_ID,
        "seed": SEED, "maxSeqLength": MAX_SEQ_LENGTH, "enableThinking": False,
        "inferenceVariant": INFERENCE_VARIANT, "loadIn4bit": True,
        "trainingConfig": {"rank": 16, "alpha": 16, "dropout": 0, "epochs": 1, "batchSize": 1, "gradientAccumulation": 8, "learningRate": 2e-4, "completionOnlyLoss": True, "packing": False},
        "dataset": DATA_REPORT, "python": platform.python_version(), "packages": packages,
        "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),
        "candidateRequirementsSha256": preflight.sha256_file(BASE / "training/requirements-colab.txt"),
    }
    (RUN_DIR / "environment.freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")
    (RUN_DIR / "run-manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    if RUN_TRAINING:
        model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL_ID, revision=MODEL_REVISION, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True, dtype=None)
        model = FastLanguageModel.get_peft_model(model, r=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], lora_alpha=16, lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth", random_state=SEED, use_rslora=False, loftq_config=None)
    elif INFERENCE_VARIANT == "BASE":
        model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL_ID, revision=MODEL_REVISION, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True, dtype=None)
        loaded_base_model = True
    else:
        if ADAPTER_PATH is None:
            raise ValueError("Set ADAPTER_PATH to a real local adapter directory for prediction/export without training.")
        ADAPTER_PATH = Path(ADAPTER_PATH).resolve()
        if not (ADAPTER_PATH / "adapter_config.json").is_file() or not any(ADAPTER_PATH.glob("adapter_model.*")):
            raise FileNotFoundError("The selected path does not contain saved adapter weights and adapter_config.json.")
        adapter_config = json.loads((ADAPTER_PATH / "adapter_config.json").read_text(encoding="utf-8"))
        if adapter_config.get("base_model_name_or_path") != MODEL_ID:
            raise ValueError("Adapter base model differs from the declared Qwen3-4B target.")
        if adapter_config.get("revision") != MODEL_REVISION:
            raise ValueError("Adapter must record the exact matching base revision; do not load an unpinned base implicitly.")
        model, tokenizer = FastLanguageModel.from_pretrained(model_name=str(ADAPTER_PATH), revision=MODEL_REVISION, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True, dtype=None)
        loaded_existing_adapter = True
        manifest["adapterPathName"] = ADAPTER_PATH.name
        manifest["adapterConfigSha256"] = preflight.sha256_file(ADAPTER_PATH / "adapter_config.json")
    if tokenizer.eos_token != "<|im_end|>":
        raise ValueError("Unexpected EOS token: recheck Qwen3 checkpoint/template compatibility.")
    manifest["chatTemplateSha256"] = hashlib.sha256(tokenizer.chat_template.encode()).hexdigest()
    manifest["eosToken"] = tokenizer.eos_token
    (RUN_DIR / "run-manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
else:
    print("Model loading and all CUDA/model actions disabled.")


## Prompt/completion preparation and supervised-token inspection

The source JSONL stays in the shared `messages` format. For TRL 0.22.2 it is converted to a standard prompt/completion dataset. Qwen's empty thinking prefix is prefilled in the prompt and masked; only the final JSON plus EOS is supervised. No chain-of-thought corpus is required for this direct decision task. Evaluation records are never passed to the trainer. Every tokenized train/validation example must fit in full; exceeding the limit is an error, not silent truncation.


In [ ]:
def read_records(split):
    return [json.loads(line) for line in (BASE / "data" / f"{split}.jsonl").read_text(encoding="utf-8").splitlines()]

def render_training_record(record):
    prompt = tokenizer.apply_chat_template(record["messages"][:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    completion = record["messages"][-1]["content"] + tokenizer.eos_token
    token_ids = tokenizer(prompt + completion, add_special_tokens=False)["input_ids"]
    if len(token_ids) > MAX_SEQ_LENGTH:
        raise ValueError(f"{record['id']} has {len(token_ids)} tokens; reduce fixture context or deliberately raise MAX_SEQ_LENGTH.")
    return {"prompt": prompt, "completion": completion}

if RUN_TRAINING:
    from datasets import Dataset
    from trl import SFTConfig, SFTTrainer
    train_data = Dataset.from_list([render_training_record(row) for row in read_records("train")])
    validation_data = Dataset.from_list([render_training_record(row) for row in read_records("validation")])
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=train_data, eval_dataset=validation_data, args=SFTConfig(
        output_dir=str(RUN_DIR / "checkpoints"), max_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1, gradient_accumulation_steps=8, per_device_eval_batch_size=1,
        num_train_epochs=1, learning_rate=2e-4, warmup_ratio=0.1,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(), optim="adamw_8bit",
        weight_decay=0.01, lr_scheduler_type="linear", logging_steps=1, seed=SEED,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        completion_only_loss=True, packing=False, padding_free=False,
        eos_token=tokenizer.eos_token, report_to="none",
    ))
    # Inspect labels produced by the actual installed stack before any optimizer step.
    for label, dataset in (("train", trainer.train_dataset), ("validation", trainer.eval_dataset)):
        probe = trainer.data_collator([dataset[0]])
        labels = probe["labels"][0].tolist()
        supervised = [token for token in labels if token != -100]
        if not supervised or not any(token == -100 for token in labels):
            raise ValueError(f"{label}: missing completion labels or prompt masking.")
        supervised_text = tokenizer.decode(supervised, skip_special_tokens=False).strip()
        if "<think>" in supervised_text or not supervised_text.endswith(tokenizer.eos_token):
            raise ValueError(f"{label}: unexpected thinking labels or missing EOS; inspect the installed stack.")
        json.loads(supervised_text[:-len(tokenizer.eos_token)].strip())
        print(label, "supervised JSON/EOS preview:", supervised_text)
else:
    print("Trainer preparation disabled.")


## Manual SFT and validation loss

This is a short starting configuration, not tuned hyperparameters. The synthetic sample corpus is too small for a quality claim. Expand and review grouped data before a real experiment. The validation split selects/checks training; the held-out evaluation split stays separate. Validation loss alone does not establish grounded recommendations or safety behavior.


In [ ]:
if RUN_TRAINING:
    train_result = trainer.train()
    validation_metrics = trainer.evaluate()
    adapter_dir = RUN_DIR / "adapter"
    for config in model.peft_config.values():
        config.base_model_name_or_path = MODEL_ID
        config.revision = MODEL_REVISION
    model.save_pretrained(str(adapter_dir))
    tokenizer.save_pretrained(str(adapter_dir))
    trained_in_this_session = True
    manifest["status"] = "trained-awaiting-held-out-evaluation"
    manifest["trainingMetrics"] = train_result.metrics
    manifest["validationMetrics"] = validation_metrics
    (RUN_DIR / "run-manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Adapter and tokenizer saved locally. No upload performed.")
else:
    print("Training disabled.")


## Optional held-out predictions for the TypeScript evaluator

Enable `GENERATE_PREDICTIONS` only when intentionally running the loaded adapter. Each output row is `{id, output}`; `output` is the raw generated string, including malformed output so the evaluator can count failures. The generation prompt uses only system/user messages. The gold assistant message is never copied into predictions. No HTTP/Ollama client is included here. Run the repository's TypeScript evaluator against the resulting file; compare base/adapter/GGUF separately with declared model metadata before claiming improvement.


In [ ]:
def write_predictions(path, predictions, case_ids):
    seen = set()
    for row in predictions:
        if set(row) != {"id", "output"} or row["id"] not in case_ids or row["id"] in seen:
            raise ValueError("Unknown/duplicate prediction id or invalid prediction keys.")
        seen.add(row["id"])
    if seen != set(case_ids):
        raise ValueError("Missing predictions; do not silently omit failed cases.")
    Path(path).write_text("".join(json.dumps(row, ensure_ascii=False, separators=(",", ":")) + "\n" for row in predictions), encoding="utf-8")

if GENERATE_PREDICTIONS:
    if not trained_in_this_session and not loaded_existing_adapter and not loaded_base_model:
        raise RuntimeError("Prediction requires an explicitly loaded BASE or a trained/loaded ADAPTER.")
    FastLanguageModel.for_inference(model)
    evaluation_records = read_records("evaluation")
    predictions = []
    for record in evaluation_records:
        # Deliberately exclude the gold assistant message from every generation.
        prompt = tokenizer.apply_chat_template(record["messages"][:2], tokenize=False, add_generation_prompt=True, enable_thinking=False)
        encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
        if encoded["input_ids"].shape[-1] + 512 > MAX_SEQ_LENGTH:
            raise ValueError(f"{record['id']}: prompt plus output reserve exceeds context; resize deliberately.")
        encoded = {key: value.to("cuda") for key, value in encoded.items()}
        with torch.inference_mode():
            generated = model.generate(**encoded, max_new_tokens=512, do_sample=False, use_cache=True, eos_token_id=tokenizer.eos_token_id)
        raw_output = tokenizer.decode(generated[0, encoded["input_ids"].shape[-1]:], skip_special_tokens=True)
        predictions.append({"id": record["id"], "output": raw_output})
    write_predictions(RUN_DIR / "predictions.jsonl", predictions, [row["id"] for row in evaluation_records])
    manifest["predictionFile"] = "predictions.jsonl"
    manifest["predictionSha256"] = preflight.sha256_file(RUN_DIR / "predictions.jsonl")
    manifest["predictionCases"] = len(predictions)
    (RUN_DIR / "run-manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Raw held-out predictions saved for external evaluation. Quality has not been inferred here.")
else:
    print("Prediction generation disabled.")


## Optional merged GGUF export

Direct adapter import is not the serving plan: current upstream Ollama rejects LoRA adapters. Keep the adapter/tokenizer as experiment artifacts; merge to 16 bit and export a complete GGUF. Export can require more RAM/disk than QLoRA training. Preserve the training template/EOS and test exported predictions again. Ollama does not quantize a GGUF during import. The example Modelfile expects the exported file to be named `merged-q4_k_m.gguf`; inspect the actual output filename before copying it.


In [ ]:
if RUN_EXPORT:
    if not trained_in_this_session and not loaded_existing_adapter:
        raise RuntimeError("Export requires a trained adapter, never the untouched base model.")
    export_dir = RUN_DIR / "export"
    export_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained_merged(str(export_dir / "merged-16bit"), tokenizer, save_method="merged_16bit", maximum_memory_usage=0.5)
    model.save_pretrained_gguf(str(export_dir / "gguf"), tokenizer, quantization_method="q4_k_m")
    manifest["status"] = "exported-awaiting-ollama-parity-check"
    manifest["export"] = {"merge": "merged_16bit", "quantization": "q4_k_m", "parityChecked": False}
    (RUN_DIR / "run-manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Inspect actual GGUF filename and template metadata; run the same held-out evaluator after serving.")
else:
    print("Export disabled.")


## Official references and handoff

- [Qwen3-4B model card](https://huggingface.co/Qwen/Qwen3-4B): model identity, minimum Transformers version and thinking control.
- [Qwen tokenizer template](https://huggingface.co/Qwen/Qwen3-4B/raw/main/tokenizer_config.json): empty non-thinking prefix and EOS.
- [Unsloth Qwen3 notebook](https://github.com/unslothai/notebooks/blob/main/nb/Qwen3_(14B)-Reasoning-Conversational.ipynb): installation profile and LoRA example; adapt the model size and data deliberately.
- [TRL 0.22.2 SFT](https://huggingface.co/docs/trl/v0.22.2/en/sft_trainer): prompt/completion data and completion-only loss.
- [Unsloth GGUF export](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf): merge, quantization, template/EOS parity.
- [Ollama current import documentation](https://docs.ollama.com/import), [upstream adapter rejection](https://github.com/ollama/ollama/blob/main/server/create.go), [structured outputs](https://docs.ollama.com/capabilities/structured-outputs), [thinking](https://docs.ollama.com/capabilities/thinking).

Future application request: JSON Schema from `assistantDecisionSchema`, `think: false`, bounded context/output and timeout; parse with `parseAssistantDecision`, then recheck IDs, availability, requested categories and integer budget against current database state. A schema does not prove a recommendation is factually valid. The application remains responsible for authoritative prices and mutations.
